![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 7 — FinOps, Presupuestos y Gobierno de Costos
**Rol:** CRB_INFRAESTRUCTURA | **Tiempo:** 15 min | **Criterio RFP:** Administración, DataOps, FinOps

> **Para la audiencia financiera:** Este track demuestra cómo Snowflake permite visibilidad total del consumo, atribución de costos por dominio de negocio (chargeback), presupuestos con alertas automáticas, detección de anomalías en el gasto, y gobernanza de costos — todo nativo, sin herramientas externas.

In [ ]:
USE ROLE CRB_INFRAESTRUCTURA;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;
USE SCHEMA PLATAFORMA;

## Bloque 1 — Visibilidad Total del Consumo
Snowflake expone **100% del consumo** a través de vistas nativas en `SNOWFLAKE.ACCOUNT_USAGE`. No se necesitan herramientas externas para saber cuánto gasta cada equipo.

In [ ]:
-- 1.1 Consumo por servicio: Compute, Cloud Services, Serverless
-- Esta es la vista que un CFO necesita: ¿en qué se gastan los créditos?
SELECT
    SERVICE_TYPE,
    SUM(CREDITS_USED_COMPUTE)::NUMBER(10,2)         AS CREDITOS_COMPUTE,
    SUM(CREDITS_USED_CLOUD_SERVICES)::NUMBER(10,2)  AS CREDITOS_CLOUD_SVC,
    SUM(CREDITS_USED)::NUMBER(10,2)                  AS CREDITOS_TOTALES
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY 1
ORDER BY 4 DESC;

La tendencia diaria revela patrones de uso y picos inesperados. Esto alimenta alertas proactivas y planificación de capacidad.

In [ ]:
-- 1.2 Tendencia diaria de consumo (detectar picos)
SELECT
    USAGE_DATE                                       AS FECHA,
    SUM(CREDITS_USED)::NUMBER(10,2)                  AS CREDITOS_DIA,
    AVG(SUM(CREDITS_USED)) OVER (
        ORDER BY USAGE_DATE ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    )::NUMBER(10,2)                                   AS PROMEDIO_MOVIL_7D
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY 1
ORDER BY 1;

Identificamos las queries más costosas de la semana — candidatas a optimización inmediata. Snowflake expone métricas de performance (tiempo, GB escaneados, particiones) para cada ejecución.

In [ ]:
-- 1.3 Top 10 queries más costosas de la semana
-- Estas son las candidatas a optimización inmediata
SELECT
    QUERY_ID,
    LEFT(QUERY_TEXT, 80)                             AS PREVIEW,
    WAREHOUSE_NAME,
    USER_NAME,
    ROUND(TOTAL_ELAPSED_TIME / 1000, 1)             AS SEGUNDOS,
    ROUND(BYTES_SCANNED / 1e9, 2)                   AS GB_ESCANEADOS,
    PARTITIONS_SCANNED || '/' || PARTITIONS_TOTAL    AS PARTICIONES
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME IS NOT NULL
ORDER BY TOTAL_ELAPSED_TIME DESC
LIMIT 10;

## Bloque 2 — Etiquetado FinOps y Chargeback por Dominio
Los **Tags** permiten clasificar warehouses, tablas y schemas por centro de costo. Esto habilita reportes de chargeback automáticos.

In [ ]:
-- 2.1 Ver tags disponibles (ya creados por el setup)
SHOW TAGS IN SCHEMA CREDIBANCO_HOL.GOBIERNO;

Etiquetamos recursos por dominio de negocio. Los tags habilitan **chargeback automático** — cada área paga solo por lo que consume, sin asignaciones manuales.

In [ ]:
-- 2.2 Etiquetar recursos por dominio de negocio
ALTER WAREHOUSE CREDIBANCO_HOL_WH
  SET TAG CREDIBANCO_HOL.GOBIERNO.TAG_DOMINIO = 'PLATAFORMA';

ALTER TABLE CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
  SET TAG CREDIBANCO_HOL.GOBIERNO.TAG_DOMINIO = 'PAGOS';

ALTER TABLE CREDIBANCO_HOL.PAGOS.LIQUIDACIONES
  SET TAG CREDIBANCO_HOL.GOBIERNO.TAG_DOMINIO = 'PAGOS';

El reporte de **chargeback** cruza consumo real con tags de dominio. Cada área de CredibanCo ve exactamente cuánto consume — sin estimaciones.

In [ ]:
-- 2.3 Reporte de chargeback: consumo cruzado con tags
-- Esto es lo que el área financiera necesita para facturación interna
SELECT
    tr.TAG_VALUE                            AS CENTRO_COSTO,
    wmh.WAREHOUSE_NAME,
    SUM(wmh.CREDITS_USED)::NUMBER(10,4)    AS CREDITOS_TOTALES,
    COUNT(*)                                AS REGISTROS_METERING
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY wmh
JOIN TABLE(CREDIBANCO_HOL.INFORMATION_SCHEMA.TAG_REFERENCES(
    'CREDIBANCO_HOL_WH', 'WAREHOUSE'
)) tr ON tr.TAG_NAME = 'TAG_DOMINIO'
WHERE wmh.WAREHOUSE_NAME = 'CREDIBANCO_HOL_WH'
  AND wmh.START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY 3 DESC;

## Bloque 3 — Resource Monitors y Alertas Automáticas
Snowflake permite definir **presupuestos** (Resource Monitors) y **alertas** que actúan automáticamente cuando se superan umbrales.

In [ ]:
-- 3.1 Ver presupuestos (Resource Monitors) configurados
SHOW RESOURCE MONITORS;

Las alertas proactivas detectan anomalías de consumo. Si un warehouse excede umbrales definidos, Snowflake notifica automáticamente — prevención antes de que el gasto se dispare.

In [ ]:
-- 3.2 Crear alerta proactiva: consumo anómalo (>2x promedio horario)
CREATE OR REPLACE ALERT CREDIBANCO_HOL.PLATAFORMA.ALERTA_CONSUMO_ANOMALO_USER
  WAREHOUSE = CREDIBANCO_HOL_WH
  SCHEDULE = '60 MINUTE'
  IF (EXISTS (
    SELECT 1
    FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
    WHERE START_TIME >= DATEADD('hour', -1, CURRENT_TIMESTAMP())
      AND CREDITS_USED > 2
  ))
  THEN
    SELECT 'ALERTA: Consumo anómalo detectado — revisar warehouse' AS MENSAJE;

-- Activar la alerta
ALTER ALERT CREDIBANCO_HOL.PLATAFORMA.ALERTA_CONSUMO_ANOMALO_USER RESUME;

-- Verificar alertas activas
SHOW ALERTS IN SCHEMA CREDIBANCO_HOL.PLATAFORMA;

## Bloque 4 — Detección de Anomalías en el Gasto
Comparamos el consumo reciente contra el promedio histórico para identificar desviaciones que requieren investigación.

Detección estadística de anomalías: comparamos consumo diario contra el promedio de 90 días usando Z-score. Días con gasto > 2 desviaciones estándar se marcan como anomalías para investigación.

In [ ]:
-- 4.1 Detección de anomalías: días con consumo > 2 desviaciones estándar
WITH daily_usage AS (
    SELECT
        USAGE_DATE,
        SUM(CREDITS_USED) AS credits
    FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
    WHERE USAGE_DATE >= DATEADD('day', -90, CURRENT_DATE())
    GROUP BY 1
),
stats AS (
    SELECT
        AVG(credits)::NUMBER(10,2) AS avg_credits,
        STDDEV(credits)::NUMBER(10,4) AS stddev_credits
    FROM daily_usage
)
SELECT
    d.USAGE_DATE                                     AS FECHA,
    d.credits::NUMBER(10,2)                          AS CREDITOS_DIA,
    s.avg_credits                                    AS PROMEDIO_90D,
    s.stddev_credits                                 AS DESV_ESTANDAR,
    ROUND((d.credits - s.avg_credits) / NULLIF(s.stddev_credits, 0), 2) AS Z_SCORE,
    CASE
        WHEN (d.credits - s.avg_credits) / NULLIF(s.stddev_credits, 0) > 2 THEN 'ANOMALIA ALTA'
        WHEN (d.credits - s.avg_credits) / NULLIF(s.stddev_credits, 0) > 1.5 THEN 'ATENCION'
        ELSE 'NORMAL'
    END                                              AS CLASIFICACION
FROM daily_usage d
CROSS JOIN stats s
ORDER BY d.USAGE_DATE DESC
LIMIT 30;

## Bloque 5 — CoCo: Dashboard FinOps en React

Copia este prompt en **Cortex Code** para generar un dashboard local de seguimiento de costos:

> **Genera un dashboard FinOps en React con los siguientes componentes:
> 1. KPIs principales: créditos totales (30d), tendencia vs mes anterior (%), warehouse más costoso
> 2. Gráfico de líneas: tendencia diaria de consumo con línea de promedio móvil 7 días
> 3. Gráfico de barras apiladas: consumo por servicio (Compute, Cloud Services, Serverless)
> 4. Tabla de chargeback: consumo por centro de costo/dominio usando tags
> 5. Sección de anomalías: días con Z-score > 2 marcados en rojo
> 6. Gráfico de dona: distribución del gasto por warehouse
> Los datos vienen de SNOWFLAKE.ACCOUNT_USAGE (METERING_DAILY_HISTORY, WAREHOUSE_METERING_HISTORY, QUERY_HISTORY). Usa Recharts para gráficos y Tailwind para estilo. Incluye filtro de rango de fechas.**

In [ ]:
-- Verificación final
SELECT 'T7_FINOPS_COMPLETO' AS status,
       CURRENT_TIMESTAMP() AS completado_en;